# Lakeside heating DSM — PyTorch deep bake-off (experimental)

Creative tabular nets on the **native** EnergyPlus farm from the site Lakeside
staged utility champion. Writes `heating_dsm_hourly_torch_v1.onnx` only.

**Slim bake-off** (after full architecture search): keep only contenders.

| Family | Why kept |
|---|---|
| `resmlp` | Bake-off winner (~21.5 kW peak MAE — edges sklearn GB) |
| `gated_mlp` | Close runner-up |
| `mlp` | Cheap baseline |

Dropped: CatBoost notebook, hour_cnn, wide_deep, hour_aware, FT-Transformer, pinball.


## 0 · Setup


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path("..").resolve()
if not (ROOT / "ml").is_dir():
    ROOT = Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "ml"))

from artifact_paths import artifact_paths, train_parquet_path
from feature_compile_heating_dsm import assert_no_future_leakage, compile_features, matrix_xy, morning_peak_mask
from notebook_plots import family_mae_bars
from notebook_proof import prove_native_farm_load

PATHS = artifact_paths()
SITE = Path(os.environ.get("LAKESIDE_SITE_ROOT", r"C:\Users\ben\OneDrive\Desktop\testing\sp_creekside"))
print("ROOT", ROOT)
print("SITE", SITE)
print("farm parquet", PATHS["eplus_farm"])

import torch
from train_heating_dsm_torch import bake_off_torch, export_onnx, load_gb_ship_peak_mae
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device", DEVICE)


## 1 · Proof — load native farm parquet (human visible)


In [ ]:
df_raw, proof = prove_native_farm_load(root=ROOT, paths=PATHS, site=SITE)
# Fail closed: train path must resolve to the same native farm
assert train_parquet_path().resolve() == PATHS["eplus_farm"].resolve()
print("train_parquet_path OK →", train_parquet_path())


## 2 · Compile features + leakage check


In [ ]:
feat = compile_features(df_raw)
assert_no_future_leakage(feat)
X, y, groups, cols = matrix_xy(feat)
peak = morning_peak_mask(feat)
df = feat
print(f"rows={len(feat)} days={feat['day'].nunique()} features={len(cols)} peak_hours={int(peak.sum())}")
show = [c for c in ("day", "hour_ending", "facility_kw", "oat_f", "strategy_id", "provenance") if c in feat.columns]
display(feat[show].head(12))


## 3 · Architecture bake-off


In [ ]:
EPOCHS = 45
N_SPLITS = 5
gb_peak = load_gb_ship_peak_mae()
result = bake_off_torch(df, n_splits=N_SPLITS, epochs=EPOCHS, device=DEVICE, include_quantile=True)
cv = result["cv"]
display(pd.DataFrame(cv).T.sort_values("mae_peak_05_09"))
print("champion:", result["champion"])

fig, ax = plt.subplots(figsize=(9, 4.5))
family_mae_bars(cv, ax=ax, title="PyTorch bake-off · peak HE 05–09", highlight=result["champion"])
if gb_peak is not None:
    ax.axvline(gb_peak, color="#54A24B", ls="--", label=f"GB ship {gb_peak:.1f}")
    ax.legend(loc="lower right")
plt.tight_layout()
plt.show()


## 4 · Export torch alt ONNX + compare to GB


In [ ]:
art = PATHS["onnx"].parent
onnx_path = art / "heating_dsm_hourly_torch_v1.onnx"
meta_path = art / "heating_dsm_hourly_torch_v1_feature_meta.json"
export_onnx(result["model"], result["n_in"], onnx_path, device="cpu")
meta = {
    "feature_cols": result["feature_cols"],
    "scaler_mean": result["scaler"].mean_.tolist(),
    "scaler_scale": result["scaler"].scale_.tolist(),
    "champion": result["champion"],
    "family": "pytorch",
    "cv": result["cv"],
    "family_loss": result["family_loss"],
    "schema": "lakeside.heating_dsm_hourly.torch_v1",
    "training_source": "ENERGYPLUS_NATIVE_RUN",
    "cv_mae_peak_05_09": result["cv"][result["champion"]]["mae_peak_05_09"],
    "gb_ship_mae_peak_05_09": gb_peak,
    "honesty": "PyTorch CANDIDATE on ENERGYPLUS_NATIVE_RUN (site Lakeside staged twin). Not desktop ship.",
}
meta_path.write_text(json.dumps(meta, indent=2) + "\n", encoding="utf-8")

import onnxruntime as ort
Xs = result["scaler"].transform(X[:8])
with torch.no_grad():
    torch_pred = result["model"].cpu()(torch.tensor(Xs, dtype=torch.float32)).numpy()
sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
onnx_pred = sess.run(None, {"features": Xs.astype(np.float32)})[0].reshape(-1)
print("onnx roundtrip max_abs", float(np.max(np.abs(torch_pred - onnx_pred))))
torch_peak = result["cv"][result["champion"]]["mae_peak_05_09"]
delta = (gb_peak - torch_peak) if gb_peak is not None else None
display(Markdown(
    f"### Cross-compare\n"
    f"- Torch **{result['champion']}** peak MAE = **{torch_peak:.2f} kW**\n"
    + (f"- GB ship peak MAE = **{gb_peak:.2f} kW** (Δ={delta:+.2f})\n" if gb_peak else "")
    + "- Desktop v1 untouched."
))
